# Spotify Project Data Cleaning

**Student:** Zhouyan "Hazel" Yuan  
**Project:** Track Promotion Strategy  
**Source:** Kaggle Spotify Music Features Dataset  
**Source file:** data/raw/spotify_full.csv

## A. Project Goal

This project prepares a reproducible Spotify track dataset for historical comparison within a selected genre and release period and for a downstream dashboard. The primary success metric is track_popularity; streams_total is a secondary metric and must remain clearly labeled as direct or estimated.

The cleaned data supports descriptive comparison—not causal claims or automatic hit prediction. This notebook is the lightweight final record of the audit, approved decisions, cleaning outputs, and validation evidence.

## B. File Paths and Imports

The project root is detected from the current working directory, so the notebook works when launched from either the repository root or the notebook folder. Report and sample paths first try the requested final locations and then fall back to the locations currently produced by src/clean_data.py.

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_RUN_STARTED = time.perf_counter()


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "plan.md").exists() and (candidate / "src" / "clean_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError("None of the expected paths exists: " + ", ".join(map(str, paths)))


def extract_markdown_table(text: str, heading: str) -> pd.DataFrame:
    lines = text.splitlines()
    heading_index = lines.index(heading)
    table_lines = []
    for line in lines[heading_index + 1:]:
        if line.startswith("|"):
            table_lines.append(line)
        elif table_lines:
            break
    if len(table_lines) < 2:
        raise ValueError(f"No Markdown table found after {heading!r}.")
    rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in table_lines]
    return pd.DataFrame(rows[2:], columns=rows[0])


ROOT = find_project_root(Path.cwd())
PLAN_PATH = ROOT / "plan.md"
AUDIT_PATH = ROOT / "audit_report.md"
CLEANING_SCRIPT_PATH = ROOT / "src" / "clean_data.py"
VALIDATION_PATH = ROOT / "reports" / "validation_report.md"
CLEANING_LOG_PATH = first_existing(
    ROOT / "reports" / "cleaning_log.csv",
    ROOT / "data" / "sample" / "cleaning_log.csv",
)
MANUAL_CHECKS_PATH = first_existing(
    ROOT / "reports" / "manual_record_checks.md",
    ROOT / "manual_record_checks.md",
)
SAMPLE_PATH = first_existing(
    ROOT / "data" / "processed" / "spotify_cleaned_sample.csv",
    ROOT / "data" / "sample" / "spotify_cleaned_sample.csv",
)

resolved_paths = pd.Series(
    {
        "Project root": ROOT,
        "Plan": PLAN_PATH,
        "Audit report": AUDIT_PATH,
        "Cleaning script": CLEANING_SCRIPT_PATH,
        "Cleaning log": CLEANING_LOG_PATH,
        "Validation report": VALIDATION_PATH,
        "Manual checks": MANUAL_CHECKS_PATH,
        "Cleaned sample": SAMPLE_PATH,
    },
    name="Resolved path",
).astype("string")
display(resolved_paths.to_frame())

## C. Source Data Audit Summary

The source audit below is taken from audit_report.md. The notebook does **not** load spotify_full.csv, rerun the full audit, or recompute full-data statistics.

Key observations:

- 1,182,354 rows and 83 source columns; source size 816,973,617 bytes (779.13 MiB).
- SHA-256: DD195436D5943929A1321D111B45B5560AE0EC4A743F187F58F3BF5B82D1E914.
- No exact duplicate rows, repeated track_id, repeated spotify_id, or repeated ID pairs.
- Missing fields include 5 titles, 19 artists (20-row union; 4 missing both), and 22,590 time_signature values.
- Review flags identified 407 durations over one hour and 1,228 nonpositive time_signature values. These observations do not by themselves establish errors.

In [ ]:
audit_text = AUDIT_PATH.read_text(encoding="utf-8")

audit_summary = pd.DataFrame(
    [
        ("Source rows", "1,182,354"),
        ("Source columns", "83"),
        ("File size", "779.13 MiB"),
        ("Release-year range", "1957–2023"),
        ("Exact duplicate rows", "0"),
        ("Repeated track_id / spotify_id / ID pairs", "0 / 0 / 0"),
        ("Missing title / artist / union / both", "5 / 19 / 20 / 4"),
        ("Missing time_signature", "22,590"),
        ("Duration over one hour", "407"),
        ("Nonpositive time_signature", "1,228"),
    ],
    columns=["Audit measure", "Observed result"],
)
display(audit_summary)
display(extract_markdown_table(audit_text, "## 1. Source controls"))

## D. Approved Cleaning Decisions

The final implementation follows the approved guardrails established through the plan, audit, and reviewed checks:

- Use track_id as the canonical key and spotify_id as a validation identifier.
- Retain all 1,182,354 rows and all 83 original columns; add only four derived fields.
- Convert approved text, integer, float, Boolean, and date fields to appropriate pandas types without converting missing values to zero or False.
- Trim boundary whitespace from the 30 affected track_artist values.
- Remove the single DEL control character from its title and collapse only the double space created at that boundary.
- Retain the 20 rows missing title or artist and mark them with dashboard_eligible=False.
- Retain long durations and nonpositive or missing time signatures; add duration_over_one_hour and invalid_time_signature flags.
- Add streams_status so direct and estimated streams remain distinguishable; never treat missing streams as zero.
- Preserve unusual values unless evidence proves them incorrect. Do not impute missing data.
- Preserve hit_probability for source fidelity, but exclude it from dashboard analysis and automatic hit prediction.

No source record was arbitrarily deleted, no missing value was filled, and the raw source file was not modified.

## E. Cleaning Script

The complete, reproducible, chunked cleaning implementation is maintained in src/clean_data.py; it is not duplicated here.

The safety switch below defaults to False. A normal **Run All** therefore skips the full pipeline. It will run only after a user deliberately edits the value to True.

In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    print("RUN_FULL_PIPELINE=True: starting the full cleaning pipeline.")
    subprocess.run(
        [sys.executable, str(CLEANING_SCRIPT_PATH)],
        cwd=ROOT,
        check=True,
    )
else:
    print("Full pipeline skipped (RUN_FULL_PIPELINE=False). Existing validated outputs are used.")

## F. Cleaning Results

The cleaning log contains full-data checks already produced by the pipeline. The notebook reads this small log and the deterministic 50-row cleaned sample only; it never loads the complete cleaned CSV.

In [ ]:
cleaning_log = pd.read_csv(CLEANING_LOG_PATH)
status_counts = cleaning_log["status"].value_counts(dropna=False).rename_axis("Status").to_frame("Checks")

display(status_counts)
display(cleaning_log[["rule", "check", "metric", "expected", "actual", "status"]])

assert cleaning_log["status"].eq("PASS").all(), "At least one cleaning-log check is not PASS."
print(f"All {len(cleaning_log)} cleaning-log checks passed.")

In [ ]:
sample = pd.read_csv(SAMPLE_PATH, low_memory=False)

display(pd.DataFrame({"rows": [sample.shape[0]], "columns": [sample.shape[1]]}, index=["Cleaned sample"]))
preview_columns = [
    "track_id", "title", "track_artist", "genre_l1", "release_year",
    "track_popularity", "streams_total", "streams_status",
    "dashboard_eligible", "duration_over_one_hour", "invalid_time_signature",
]
display(sample[preview_columns].head(5))

dtype_columns = [
    "track_id", "title", "release_year", "release_date", "duration_ms",
    "track_popularity", "streams_total", "streams_total_estimated",
    "dashboard_eligible", "duration_over_one_hour",
    "invalid_time_signature", "streams_status",
]
display(sample[dtype_columns].dtypes.astype(str).rename("CSV reload dtype").to_frame())

flag_summary = pd.concat(
    {
        column: sample[column].value_counts(dropna=False)
        for column in [
            "dashboard_eligible",
            "duration_over_one_hour",
            "invalid_time_signature",
            "streams_status",
        ]
    },
    names=["Field", "Value"],
).rename("Sample count").to_frame()
display(flag_summary)

print("Displayed dtypes are pandas inference after reloading the CSV sample; "
      "the cleaning script uses nullable pandas types during processing.")

## G. Validation Results

The following tables are read from the existing validation report rather than recomputed from the full datasets. They reconcile row and column counts, duplicate checks, selected missing counts, hashes, and the five reviewed manual cases.

In [ ]:
validation_text = VALIDATION_PATH.read_text(encoding="utf-8")

outcome = extract_markdown_table(validation_text, "## Outcome")
duplicate_checks = extract_markdown_table(validation_text, "## Duplicate and identifier checks")
missing_values = extract_markdown_table(validation_text, "## Missing values before and after")
selected_missing = missing_values[
    missing_values["Original column"].isin(
        ["title", "track_artist", "time_signature", "streams_source", "hit_probability"]
    )
].reset_index(drop=True)

display(Markdown("### Row, column, and hash controls"))
display(outcome)
display(Markdown("### Duplicate and identifier controls"))
display(duplicate_checks)
display(Markdown("### Selected missing-value reconciliation"))
display(selected_missing)

assert outcome["Status"].eq("PASS").all()
assert duplicate_checks["Status"].eq("PASS").all()
assert selected_missing["Difference"].eq("0").all()
print("Reported row, column, hash, duplicate, identifier, and selected missing-value controls all pass.")

In [ ]:
manual_text = MANUAL_CHECKS_PATH.read_text(encoding="utf-8")
manual_rows = []

for line in manual_text.splitlines():
    if line.startswith("|") and "spotify:track:" in line:
        row_cells = [cell.strip() for cell in line.strip("|").split("|")]
        if len(row_cells) != 14:
            raise ValueError("Unexpected manual-check table structure.")
        manual_rows.append(
            {
                "Case": row_cells[0],
                "track_id": row_cells[1].strip(chr(96)),
                "Result": row_cells[13],
            }
        )

manual_summary = pd.DataFrame(manual_rows)
display(manual_summary)

pass_count = int(manual_summary["Result"].eq("Pass").sum())
fail_count = int(manual_summary["Result"].eq("Fail").sum())
assert len(manual_summary) == 5 and pass_count == 5 and fail_count == 0
print(f"Manual checks: {pass_count} Pass, {fail_count} Fail.")
print("The approved 1e-12 tolerance treats the novelty_score CSV serialization difference as numerically equivalent.")

## H. Conclusion

The approved process retained every source row and all 83 original columns, standardized approved data types, trimmed 30 artist boundary-space values, removed one DEL control character, and added four transparent status or quality flags.

Missing and unusual values were not imputed or discarded. Long durations, nonpositive time signatures, and missing dashboard labels remain available for review and are identified through flags. Direct and estimated streams remain distinguishable.

Because the complete source and cleaned files are large, this final notebook uses the existing audit, cleaning log, validation report, manual checks, and a deterministic 50-row sample for fast, repeatable verification. The default workflow does not rerun the full pipeline.

hit_probability remains in the cleaned data for source preservation but should not be used in the final dashboard or as an automatic hit-prediction field.

In [ ]:
notebook_elapsed = time.perf_counter() - NOTEBOOK_RUN_STARTED
print(f"Lightweight notebook execution time: {notebook_elapsed:.2f} seconds")
print(f"Full cleaning pipeline executed: {RUN_FULL_PIPELINE}")